# FailurePostprocessor + VLMService.select_checkpoint_index Test

This notebook uses a real `LeRobotDataset` sample to construct a batch:
1. Initialize `LeRobotDataset` with `repo_id` + `root` and fetch one sample.
2. Read stitched PNG images from `tools/failure/examples/pick_up_markers`.
3. Split each stitched image into left/top/right views, resize to the dataset sample image size, and fill them into the batch.
4. Before calling `select_checkpoint_index`, convert batch images back to visualized images for inspection.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from lerobot.datasets.lerobot_dataset import LeRobotDataset

cwd = Path.cwd().resolve()

repo_id = "eval/eval_pick_up_markers_failure_and_ckpt_part2"
sample_index = 0
task_dir = cwd / "examples" / "pick_up_markers"
example_dirs = task_dir / "0"
cfg_path = task_dir / "failure_handling.json"

if not cfg_path.exists():
    raise FileNotFoundError(f"Missing config file: {cfg_path}")

with cfg_path.open("r", encoding="utf-8") as f:
    cfg_raw = json.load(f)

demo_video_path = Path(cfg_raw["demo_video_path"]).expanduser().resolve()
if not demo_video_path.exists():
    raise FileNotFoundError(f"demo_video_path does not exist: {demo_video_path}")

failure_root = example_dirs / "failure"
checkpoint_root = example_dirs / "checkpoints"
if not failure_root.exists():
    raise FileNotFoundError(f"Missing failure directory: {failure_root}")
if not checkpoint_root.exists():
    raise FileNotFoundError(f"Missing checkpoints directory: {checkpoint_root}")


def _collect_three_view_paths(step_dir: Path) -> dict[str, Path]:
    paths = {cam: step_dir / f"{cam}.png" for cam in ["left", "middle", "right"]}
    missing = [str(p) for p in paths.values() if not p.exists()]
    if missing:
        raise FileNotFoundError(f"time step={step_dir.name} missing view images: {missing}")
    return paths


image_paths_by_kind_step = {"failure": {}, "checkpoints": {}}
for kind, root in [("failure", failure_root), ("checkpoints", checkpoint_root)]:
    for step_dir in sorted(root.iterdir()):
        if not step_dir.is_dir():
            continue
        try:
            step = int(step_dir.name)
        except ValueError:
            continue
        image_paths_by_kind_step[kind][step] = _collect_three_view_paths(step_dir)

if not image_paths_by_kind_step["failure"]:
    raise ValueError("No failure images found.")
if not image_paths_by_kind_step["checkpoints"]:
    raise ValueError("No checkpoint images found.")

failure_steps = sorted(image_paths_by_kind_step["failure"].keys())
selected_failure_step = failure_steps[0]
checkpoint_queue = sorted(image_paths_by_kind_step["checkpoints"].keys(), reverse=True)[:5]

dataset = LeRobotDataset(repo_id, root=str(cwd.parent.parent.parent / "data" / "local"), episodes=[0])
item = dataset[sample_index]

target_hw = tuple(item["observation.images.left"].shape[-2:])  # (H, W)

VIEW_KEY_MAP = {
    "left": "observation.images.left",
    "middle": "observation.images.middle",
    "right": "observation.images.right",
}


def _pil_to_chw_float01(img: Image.Image, hw: tuple[int, int]) -> torch.Tensor:
    h, w = hw
    img = img.convert("RGB").resize((w, h), Image.BILINEAR)
    arr = np.asarray(img).astype(np.float32) / 255.0  # HWC
    arr = np.transpose(arr, (2, 0, 1))  # CHW
    return torch.from_numpy(arr)


batch_for_vlm = {k: v.unsqueeze(0).clone() if hasattr(v, "unsqueeze") else v for k, v in item.items()}

In [ ]:
from copy import deepcopy

from lerobot.policies.failure_postprocessor import FailurePostprocessor

post = FailurePostprocessor(
    policy=None,
    output_dir=None,
    failure_handling_json_path=cfg_path,
    enable_logging=False,
)

VIEW_KEY_MAP = {
    "left": "observation.images.left",
    "middle": "observation.images.top",
    "right": "observation.images.right",
}


def _to_chw_float_tensor(img_path: Path, target_hw: tuple[int, int] | None = None) -> torch.Tensor:
    img = Image.open(img_path).convert("RGB")
    if target_hw is not None:
        target_h, target_w = target_hw
        img = img.resize((target_w, target_h), Image.Resampling.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(arr).permute(2, 0, 1).contiguous()
    return tensor


def _infer_target_hw(batch_item: dict) -> tuple[int, int] | None:
    for key in VIEW_KEY_MAP.values():
        if key in batch_item and isinstance(batch_item[key], torch.Tensor):
            t = batch_item[key]
            if t.dim() == 4 and t.size(0) >= 1:
                t = t[0]
            if t.dim() == 3:
                return int(t.shape[-2]), int(t.shape[-1])
    return None


target_hw = _infer_target_hw(item)

# 1) Replace the three image tensors in the current batch with failure-step images.
failure_steps = sorted(image_paths_by_kind_step["failure"].keys())
if not failure_steps:
    raise ValueError("image_paths_by_kind_step['failure'] is empty")
selected_failure_step = failure_steps[-1]

batch_for_vlm = deepcopy(item)
for view_name, tensor_key in VIEW_KEY_MAP.items():
    img_path = image_paths_by_kind_step["failure"][selected_failure_step][view_name]
    batch_for_vlm[tensor_key] = _to_chw_float_tensor(img_path, target_hw=target_hw)

# 2) Build checkpoint_queue: (checkpoint_step, action, checkpoint_views)
checkpoint_queue = []
for step_idx in sorted(image_paths_by_kind_step["checkpoints"].keys()):
    checkpoint_views = {}
    for view_name, tensor_key in VIEW_KEY_MAP.items():
        img_path = image_paths_by_kind_step["checkpoints"][step_idx][view_name]
        checkpoint_views[tensor_key] = _to_chw_float_tensor(img_path, target_hw=target_hw)
    checkpoint_queue.append((int(step_idx), None, checkpoint_views))

print(f"selected_failure_step={selected_failure_step}")
print(f"checkpoint_queue size={len(checkpoint_queue)}")
print(f"checkpoint steps={[entry[0] for entry in checkpoint_queue]}")

In [ ]:
selected_index = post.vlm_service.select_checkpoint_index(
    batch=batch_for_vlm,
    checkpoint_queue=checkpoint_queue,
    episode=0,
    step=selected_failure_step,
)
print(f"selected_index={selected_index}, selected_step={checkpoint_queue[selected_index][0]}")

In [ ]:
saved_dir = post.vlm_service.save_debug_history()

In [ ]:
import json
from pathlib import Path

saved_path = Path(saved_dir)
manifest_path = saved_path / "manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError(f"manifest.json not found in: {saved_path}")

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

records = manifest.get("records", [])
if not records:
    raise ValueError(f"No records found in manifest: {manifest_path}")

print(f"Loaded {len(records)} record(s) from: {saved_path}")

for rec in sorted(records, key=lambda x: int(x.get("index", 0))):
    rec_index = int(rec.get("index", 0))
    rec_dir = saved_path / rec["dir"]
    meta_path = rec_dir / "meta.json"
    if not meta_path.exists():
        print(f"[Skip] missing meta.json: {rec_dir}")
        continue

    with meta_path.open("r", encoding="utf-8") as f:
        meta = json.load(f)

    parts = meta.get("request_parts", [])
    response_path = rec_dir / "response.txt"
    has_response = response_path.exists()

    total_rows = len(parts) + (1 if has_response else 0)
    fig_h = max(4, 2.2 * total_rows)
    fig, axes = plt.subplots(total_rows, 1, figsize=(14, fig_h))
    if total_rows == 1:
        axes = [axes]

    fig.suptitle(
        f"record #{rec_index} | episode={meta.get('episode')} | step={meta.get('step')} | selected={meta.get('selected_index')}",
        fontsize=12,
        y=0.995,
    )

    row = 0
    for part in parts:
        ax = axes[row]
        ax.axis("off")
        part_type = part.get("type")
        part_file = part.get("file")
        part_idx = part.get("index", row)
        part_path = rec_dir / part_file

        if part_type == "image" and part_path.exists():
            img = Image.open(part_path).convert("RGB")
            ax.imshow(img)
            ax.set_title(f"part[{part_idx}] image", loc="left", fontsize=10)
        else:
            text = (
                part_path.read_text(encoding="utf-8") if part_path.exists() else f"[Missing file] {part_file}"
            )
            ax.text(
                0.01,
                0.98,
                text,
                va="top",
                ha="left",
                wrap=True,
                fontsize=10,
                transform=ax.transAxes,
            )
            ax.set_title(f"part[{part_idx}] text", loc="left", fontsize=10)
        row += 1

    if has_response:
        ax = axes[row]
        ax.axis("off")
        response_text = response_path.read_text(encoding="utf-8")
        ax.text(
            0.01,
            0.98,
            response_text,
            va="top",
            ha="left",
            wrap=True,
            fontsize=10,
            transform=ax.transAxes,
        )
        ax.set_title("response", loc="left", fontsize=10)

    fig.tight_layout()
    plt.show()